# Compare a policy RAG agent with a LoRA fine-tune

You are deploying a library study-room assistant. A question names a branch and needs the current
reservation rule. Rules change. Compare three systems on the same cases: a frozen model, a LoRA
adapter trained on an old policy snapshot, and a RAG agent that searches the current policy text
through a read-only, MCP-shaped tool.

The capstone has two parts, and both are required.

- **Offline core (sections 1 to 6, any laptop, standard library only).** The retriever is real:
  BM25 over chunked policy text, as taught in C4 (represent text) and C5
  (combine retrievers).
  The two model stand-ins are deliberately tiny so every number can be checked by hand. The
  "frozen model" and "LoRA" are a five-feature, two-logit toy with a rank-one adapter, and the RAG
  agent's reading step is a deterministic keyword reader (`read_rule`), not an LLM. This part
  fixes the eval suite, the scoring and the expected behaviour of each system.
- **GPU milestone (section 7, free Colab T4, about 30 to 45 minutes).** The stand-ins are replaced
  by a real open instruct model (Qwen2.5-0.5B-Instruct). You fine-tune it with LoRA (`peft`) on
  the same old snapshot and run the same eval suite on the frozen base model, the LoRA model and
  RAG with the base model (retrieved chunks in the prompt).

Complete the TODO cells, run the assertions and write your release rule before looking at the
holdout report. No API keys are needed anywhere.


In [ ]:
import math
import re
from collections import Counter, defaultdict

BRANCHES = ("north", "south", "east", "west", "central")
LABELS = ("walk_in", "reservation")
# The old snapshot the fine-tune learns from. East and West changed after it was taken.
OLD = {"north": "walk_in", "south": "reservation", "east": "walk_in", "west": "reservation"}

# Current policy documents. Each paragraph becomes one retrievable chunk.
DOCS = [
    {"id":"north-1","branch":"north","topic":"study_room","revision":1,"public":True,
     "text":"North branch study rooms are walk-in only. Take any free room for up to two hours.\n\n"
            "North branch study rooms have whiteboards and a power socket at every desk."},
    {"id":"south-1","branch":"south","topic":"study_room","revision":1,"public":True,
     "text":"South branch study rooms need a reservation. Book a room online up to a week ahead.\n\n"
            "South branch study rooms seat up to six people."},
    {"id":"east-1","branch":"east","topic":"study_room","revision":1,"public":True,
     "text":"East branch study rooms are walk-in. Rooms are first come, first served.\n\n"
            "East branch study rooms close thirty minutes before the library."},
    {"id":"east-2","branch":"east","topic":"study_room","revision":2,"public":True,
     "text":"Revision 2. East branch study rooms now need a reservation. Book online before you arrive.\n\n"
            "East branch study rooms close thirty minutes before the library."},
    {"id":"west-1","branch":"west","topic":"study_room","revision":1,"public":True,
     "text":"West branch study rooms need a reservation made at the front desk.\n\n"
            "West branch study rooms are on the second floor."},
    {"id":"west-2","branch":"west","topic":"study_room","revision":2,"public":True,
     "text":"Revision 2. West branch study rooms are now walk-in. No booking is needed.\n\n"
            "West branch study rooms are on the second floor."},
    {"id":"central-1","branch":"central","topic":"study_room","revision":1,"public":True,
     "text":"Central branch study rooms need a reservation through the library app.\n\n"
            "Central branch has twelve study rooms on the third floor."},
    {"id":"annex-1","branch":"annex","topic":"study_room","revision":1,"public":True,
     "text":"Annex branch study rooms are walk-in.\n\nThe annex branch is open on weekends only."},
    {"id":"all-rooms","branch":"all","topic":"study_room","revision":1,"public":True,
     "text":"Study rooms at every branch are for quiet group study. Food is not allowed in study rooms."},
    {"id":"north-99","branch":"north","topic":"study_room","revision":99,"public":False,
     "text":"Draft. North branch study rooms will need a reservation."},
    {"id":"north-hours","branch":"north","topic":"opening_hours","revision":3,"public":True,
     "text":"North branch opening hours are 09:00 to 20:00 on weekdays.\n\n"
            "The north branch is closed on public holidays."},
]

def chunk_documents(docs):
    # One chunk per paragraph. Each chunk keeps its document's metadata and a stable ID.
    return [{"id":f"{doc['id']}#{n}", "source":doc["id"], "branch":doc["branch"], "topic":doc["topic"],
             "revision":doc["revision"], "public":doc["public"], "text":paragraph.strip()}
            for doc in docs for n, paragraph in enumerate(doc["text"].split("\n\n"), 1)]

CHUNKS = chunk_documents(DOCS)
STOPWORDS = set("a an and are at be can do does for how i in is it my of on or the to what when where with".split())

def terms(text):
    return [t for t in re.findall(r"[a-z0-9]+", text.casefold()) if t not in STOPWORDS]

def read_rule(text):
    # Offline stand-in for the LLM reading step: a chunk states a rule only if it names exactly one.
    found = {label for key, label in (("walk-in","walk_in"), ("reservation","reservation"))
             if key in text.casefold()}
    return found.pop() if len(found) == 1 else None

# (case ID, question, branch, topic, expected label or None, current source ID, slice)
# Development and holdout cases use disjoint branch-topic pairs.
DEV = [
    ("d1","Are Annex branch study rooms walk-in?","annex","study_room","walk_in","annex-1","new_branch"),
    ("d2","Do I need to book a study room at the Missing branch?","missing","study_room",None,None,"no_policy"),
]

HOLDOUT = [
    ("h1","Do I need to book a study room at the North branch?","north","study_room","walk_in","north-1","stable"),
    ("h2","Do I need to book a study room at the South branch?","south","study_room","reservation","south-1","stable"),
    ("h3","Can I walk in to a study room at the East branch or must I book?","east","study_room","reservation","east-2","changed"),
    ("h4","Is a reservation needed for West branch study rooms?","west","study_room","walk_in","west-2","changed"),
    ("h5","How do study rooms work at the Central branch?","central","study_room","reservation","central-1","new_branch"),
    ("h6","Do I need to book a study room at the Harbour branch?","harbour","study_room",None,None,"no_policy"),
    ("h7","Where can I park at the North branch?","north","parking",None,None,"no_policy"),
]

assert {(branch, topic) for _, _, branch, topic, *_ in DEV}.isdisjoint(
    (branch, topic) for _, _, branch, topic, *_ in HOLDOUT
)

def encode(branch, topic):
    return [float(topic == "study_room" and branch == known) for known in BRANCHES]

def verdict(label, citation=None):
    return {"action":"abstain" if label is None else "answer",
            "label":label, "citation":citation}

print(len(DOCS), "documents,", len(CHUNKS), "chunks")


## 1. Rank policy chunks with BM25 (the read-only tool)

Implement `bm25_scores(query_terms, docs_terms, k1=1.2, b=0.75)` with the C4 formula. `docs_terms`
is a list of token lists. Return one float per document, in order, with
$\text{idf}_t = \ln\!\left(1 + \frac{N - \text{df}_t + 0.5}{\text{df}_t + 0.5}\right)$, and count each
distinct query term once. Return `[]` for an empty corpus.

Then implement `policy_search(query, k=3)`. Rank only public chunks, scoring `terms(query)` against
`terms(chunk["text"])`. Drop chunks that score 0, sort by decreasing score then ascending chunk ID,
and return at most `k` chunks projected to `id, source, branch, topic, revision, text`. The
trusted application fixes the corpus and the public-only rule: a query cannot widen it.


In [ ]:
def bm25_scores(query_terms, docs_terms, k1=1.2, b=0.75):
    raise NotImplementedError

def policy_search(query, k=3):
    raise NotImplementedError


In [ ]:
toy = [["card","refund","card"], ["card"], ["refund","policy","card","x"], ["other"]]
expected = [1.0584, 0.4616, 0.7964, 0.0]
assert all(math.isclose(got, want, abs_tol=1e-4) for got, want in zip(bm25_scores(["card","refund"], toy), expected)), bm25_scores(["card","refund"], toy)
assert bm25_scores(["card","card"], toy) == bm25_scores(["card"], toy), "Count each distinct query term once"
assert bm25_scores(["card"], []) == []

hits = policy_search("Do I need to book a study room at the North branch?")
assert [row["id"] for row in hits] == ["south-1#1","north-1#1","east-2#1"], [row["id"] for row in hits]
assert set(hits[0]) == {"id","source","branch","topic","revision","text"}
assert len(policy_search("East branch study rooms", 1)) == 1
assert all(row["source"] != "north-99" for row in policy_search("draft north reservation", 20)), "Never return private chunks"
assert policy_search("parking garage") == [], "Drop chunks that score 0"
print("OK")


In [ ]:
def send_fixture(request):
    # Return an MCP-shaped reply without starting a server process.
    params = request["params"]
    if request["method"] != "tools/call" or params["name"] != "policy_search":
        return {"jsonrpc":"2.0","id":request["id"],"error":{"code":-32602,"message":"Unsupported tool"}}
    args = params["arguments"]
    records = policy_search(args["query"], args["k"])
    return {"jsonrpc":"2.0","id":request["id"],"result":{"structuredContent":{"records":records}}}


## 2. Execute one bounded, grounded request

Implement `rag_agent(question, branch, topic, send, max_reads=1, k=3)`. If `max_reads` is below 1,
abstain without calling `send`. Otherwise send exactly one `tools/call` for `policy_search` using
JSON-RPC 2.0, request ID 1 and arguments `{"query": question, "k": k}`. Abstain when there is no
reply, the ID differs, the reply has an `error` or `result.isError` is true.

Read chunks from `result.structuredContent.records`. Keep chunks whose branch and topic match the
request and for which `read_rule(text)` returns a label. Among them take the highest revision.
Abstain if the labels at that revision conflict. Otherwise return that label and cite the smallest
source ID at that revision. The agent can only be as current as what retrieval returned: if the
newest revision is not in the top `k`, it answers from an older one or abstains.


In [ ]:
def rag_agent(question, branch, topic, send, max_reads=1, k=3):
    raise NotImplementedError


In [ ]:
EAST_Q = "Can I walk in to a study room at the East branch or must I book?"
NORTH_Q = "Do I need to book a study room at the North branch?"
assert rag_agent(EAST_Q,"east","study_room",send_fixture) == verdict("reservation","east-2")
assert rag_agent(NORTH_Q,"north","study_room",send_fixture) == verdict("walk_in","north-1")
assert rag_agent("Where can I park at the North branch?","north","parking",send_fixture) == verdict(None)
assert rag_agent(NORTH_Q,"north","study_room",send_fixture,k=1) == verdict(None), "With k=1 the North chunk is not retrieved"
sent = []
rag_agent(EAST_Q,"east","study_room",lambda request: sent.append(request))
assert sent == [{"jsonrpc":"2.0","id":1,"method":"tools/call",
                 "params":{"name":"policy_search","arguments":{"query":EAST_Q,"k":3}}}], sent
assert rag_agent(EAST_Q,"east","study_room",lambda _: None) == verdict(None)
assert rag_agent(EAST_Q,"east","study_room",lambda _: {"id":2,"result":{}}) == verdict(None)
assert rag_agent(EAST_Q,"east","study_room",lambda _: {"id":1,"result":{"isError":True}}) == verdict(None)
def reply(records):
    return lambda _: {"jsonrpc":"2.0","id":1,"result":{"structuredContent":{"records":records}}}
stale = [row for row in CHUNKS if row["source"] == "east-1"]
assert rag_agent(EAST_Q,"east","study_room",reply(stale)) == verdict("walk_in","east-1"), "Answer only from what was retrieved"
both = [row for row in CHUNKS if row["source"] in ("east-1","east-2")]
assert rag_agent(EAST_Q,"east","study_room",reply(both)) == verdict("reservation","east-2"), "Use the highest revision"
conflict = [{"id":"a#1","source":"a","branch":"east","topic":"study_room","revision":2,"text":"Rooms are walk-in."},
            {"id":"b#1","source":"b","branch":"east","topic":"study_room","revision":2,"text":"Rooms need a reservation."}]
assert rag_agent(EAST_Q,"east","study_room",reply(conflict)) == verdict(None)
def unexpected(_):
    raise AssertionError("Zero read budget must make no request")
assert rag_agent(EAST_Q,"east","study_room",unexpected,0) == verdict(None)
print("OK")


## The toy frozen model and low-rank training scaffold

This section is a **math check, not an LLM**. It lets you verify the LoRA update rule and see the
stale-knowledge failure in a model small enough to inspect. Section 7 repeats the experiment with
a real 0.5B-parameter model.

For a five-feature input x, the frozen model predicts with logits [0, 0]. LoRA adds B(Ax), rank 1
and unit scaling. The scaffold trains both factors by SGD on the old snapshot. Holdouts and current
policy records never enter training. A nonzero A and zero B make the initial update zero and allow
learning. This is supervised fine-tuning, not preference training.

Unknown branches and unsupported topics have zero features. The common zero-feature policy
abstains for both base and adapter. All known study-room inputs produce a label, even if the adapter
saw no training label for that branch.


## 3. Implement the LoRA forward path


Implement logits(features, a, b). A has one row supplied as the list a and B has one column supplied
as b. The frozen base logits are both zero. Return b times the scalar dot product of a and features.
Inputs have matching dimensions, and b has two entries. A zero B must reproduce the frozen base.


In [ ]:
def logits(features, a, b):
    raise NotImplementedError


In [ ]:
assert logits([1,2],[0.5,1.5],[2,-1]) == [7,-3.5]
assert logits([1,0],[1,1],[0,0]) == [0,0]
assert logits([0,0],[1,1],[2,4]) == [0,0]

print("OK")


In [ ]:
def train_adapter(snapshot, epochs=120, learning_rate=0.3):
    a = [0.15] * len(BRANCHES)
    b = [0.0, 0.0]
    for _ in range(epochs):
        for branch, label in snapshot.items():
            features = encode(branch, "study_room")
            target = LABELS.index(label)
            scores = logits(features, a, b)
            high = max(scores)
            exps = [math.exp(value - high) for value in scores]
            gradient = [value / sum(exps) - (idx == target) for idx, value in enumerate(exps)]
            projected = sum(weight * value for weight, value in zip(a, features))
            a_gradient = sum(weight * value for weight, value in zip(b, gradient))
            a = [weight - learning_rate * a_gradient * value for weight, value in zip(a, features)]
            b = [weight - learning_rate * projected * value for weight, value in zip(b, gradient)]
    return a, b

def classify(branch, topic, a, b):
    features = encode(branch, topic)
    if not any(features):
        return verdict(None)
    scores = logits(features, a, b)
    return verdict(LABELS[max(range(len(LABELS)), key=scores.__getitem__)])

a, b = train_adapter(OLD)
assert all(classify(branch,"study_room",a,b)["label"] == label for branch, label in OLD.items())
assert classify("harbour","study_room",a,b) == verdict(None)
print("The adapter fits the old snapshot.")


## 4. Measure actions, factual labels and citations


Implement score_case(expected, source_id, answer). Return a triple of booleans representing action
correctness, factual-label correctness, and citation correctness. Expected is a label or None for
abstention. For factual correctness, require correct action and matching label. For an expected
answer, citation correctness requires factual correctness and an exact current source ID.
For an expected abstention, citation correctness requires a correct abstention with no citation.
An accurate closed-book token can pass factual grading without passing evidence grading.


In [ ]:
def score_case(expected, source_id, answer):
    raise NotImplementedError


In [ ]:
assert score_case("walk_in","n1",verdict("walk_in")) == (True,True,False)
assert score_case("walk_in","n1",verdict("walk_in","n1")) == (True,True,True)
assert score_case("walk_in","n1",verdict("reservation","n1")) == (True,False,False)
assert score_case(None,None,verdict(None)) == (True,True,True)
assert score_case(None,None,verdict("walk_in")) == (False,False,False)

print("OK")


In [ ]:
def run_report(cases, systems):
    # systems maps a name to a function (question, branch, topic) -> verdict.
    counts = defaultdict(lambda: [0,0,0,0])
    details = []
    for case_id, question, branch, topic, expected, source, tag in cases:
        for system, answer_fn in systems.items():
            answer = answer_fn(question, branch, topic)
            results = score_case(expected,source,answer)
            details.append((case_id,tag,system,answer,results))
            for slice_name in ("all", tag):
                cell = counts[(system,slice_name)]
                cell[0] += 1
                for i, passed in enumerate(results, 1):
                    cell[i] += passed
    return dict(counts), details

OFFLINE_SYSTEMS = {
    "frozen": lambda question, branch, topic: classify(branch,topic,[0.0]*len(BRANCHES),[0.0,0.0]),
    "lora": lambda question, branch, topic: classify(branch,topic,a,b),
    "rag": lambda question, branch, topic: rag_agent(question,branch,topic,send_fixture),
}

dev_counts, dev_details = run_report(DEV, OFFLINE_SYSTEMS)
assert dev_counts[("rag","all")] == [2,2,2,2]
print("Development report:", dev_counts)


## 5. Write the decision rule before the holdout

**Your answer:** Name a cost of stale policy answers and a cost of unnecessary abstention.
Write which slice or individual failure would block a release and why. Record a version for
the corpus and old training snapshot. The holdout labels are visible for teaching, so do not
tune against them. For a real project, keep the labels hidden behind a separate scorer.

Run the report below once your rule is written. Counts are attempted, action-correct,
factually correct and citation-correct. Include the denominators in your write-up.


In [ ]:
report, details = run_report(HOLDOUT, OFFLINE_SYSTEMS)
for (system, slice_name), counts in sorted(report.items()):
    print(f"{system:6} {slice_name:10} {counts}")
assert report[("rag","all")] == [7,7,7,7]
assert report[("rag","changed")] == [2,2,2,2]
assert report[("lora","stable")] == [2,2,2,0]
assert report[("lora","changed")] == [2,2,0,0]


## 6. Run the MCP transport extension

Install the v1 Python SDK in an isolated environment with `python -m pip install "mcp>=1.10,<2"`
(as of late 2026; 1.10 added structured tool output, and 2.x may change the client API). Set
`RUN_MCP = True` below to start a local subprocess, initialize a session, discover the tool and
call it on every holdout question. The server runs **your** `terms`, `bm25_scores` and
`policy_search` source code over the same chunks. The test compares each live result with the
in-memory fixture and reruns the report through the captured responses. Only the parent-created
temporary chunk file is read. Tool inputs select chunks and cannot choose a path. No API keys.

The server's stderr goes to `os.devnull`: Jupyter's `sys.stderr` is not a real file, so a
subprocess cannot write to it. Point `errlog` at a log file instead if the server fails to start.

The in-memory report measures behaviour only. To compare latency, warm the server and
measure repeated calls separately from initialization.


In [ ]:
RUN_MCP = False

async def collect_live_responses(cases, k=3):
    import inspect
    import json
    import os
    import sys
    from pathlib import Path
    from tempfile import TemporaryDirectory

    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    learner_code = "\n\n".join(inspect.getsource(fn) for fn in (terms, bm25_scores, policy_search))
    server_source = f"""
import json
import math
import re
import sys
from collections import Counter, defaultdict
from pathlib import Path
from mcp.server.fastmcp import FastMCP

CHUNKS = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
STOPWORDS = set({sorted(STOPWORDS)!r})

{learner_code}

server = FastMCP("Library capstone")

# The dict[str, list] annotation makes FastMCP send structuredContent; a bare dict does not (mcp 1.x).
@server.tool(name="policy_search")
def policy_search_tool(query: str, k: int = 3) -> dict[str, list]:
    # Return the top-k public policy chunks for this query.
    return {{"records": policy_search(query, k)}}

if __name__ == "__main__":
    server.run(transport="stdio")
"""
    responses = {}
    with TemporaryDirectory(prefix="library-mcp-") as tmp, open(os.devnull, "w") as errlog:
        server_path = Path(tmp) / "server.py"
        chunk_path = Path(tmp) / "chunks.json"
        server_path.write_text(server_source, encoding="utf-8")
        chunk_path.write_text(json.dumps(CHUNKS), encoding="utf-8")
        args = StdioServerParameters(command=sys.executable, args=[str(server_path), str(chunk_path)])
        async with stdio_client(args, errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                assert "policy_search" in {tool.name for tool in tools.tools}
                for _, question, *_ in cases:
                    result = await session.call_tool("policy_search", arguments={"query": question, "k": k})
                    assert not result.isError
                    assert result.structuredContent == {"records":policy_search(question, k)}
                    responses[question] = result.structuredContent
    return responses

if RUN_MCP:
    live_responses = await collect_live_responses(HOLDOUT)

    def replay_live(request):
        args = request["params"]["arguments"]
        return {"jsonrpc":"2.0","id":request["id"],
                "result":{"structuredContent":live_responses[args["query"]]}}

    live_systems = dict(OFFLINE_SYSTEMS, rag=lambda question, branch, topic: rag_agent(question,branch,topic,replay_live))
    assert run_report(HOLDOUT, live_systems) == (report, details)
    print("Live MCP and offline reports agree.")


## 7. GPU milestone: a real LLM, a real LoRA adapter (required)

Switch Colab to a GPU runtime (Runtime > Change runtime type > T4 GPU), then run every cell above
again: this section reuses `policy_search`, `score_case`, `run_report`, `OLD` and `HOLDOUT`.
Budget about 30 to 45 minutes on a free T4, most of it installing, downloading about 1 GB of
weights and reading outputs. Free-tier GPU availability changes; Kaggle notebooks are a fallback.

The model is `Qwen/Qwen2.5-0.5B-Instruct` (Apache-2.0, about 0.5B parameters, as of late 2026).
Any instruct model under 1B parameters with a chat template works; if you swap it, check that
its attention projections are named `q_proj`, `k_proj`, `v_proj` and `o_proj`.

You will evaluate three systems on the same holdout cases and the same `score_case`:

- **base**: the frozen model, closed book.
- **lora_llm**: the same model with a LoRA adapter trained on the old snapshot only, closed book.
- **rag_base**: the frozen model with the top three `policy_search` chunks in its prompt.

Greedy decoding keeps answers repeatable on one machine, but small numeric differences across
GPUs and library versions can still flip a token. Report what you observed, not what you expected.
The next cell has no GPU code, so you can run its checks anywhere. On a machine without a GPU,
stop after it.


In [ ]:
SYSTEM = ("You answer questions about library study-room rules. "
          "Reply with exactly one of: walk-in, reservation, unknown.")

def closed_book_messages(question):
    return [{"role":"system","content":SYSTEM}, {"role":"user","content":question}]

def rag_messages(question, records):
    sources = "\n".join(f"[{row['source']}] (revision {row['revision']}, {row['branch']} branch, "
                        f"{row['topic']}) {row['text']}" for row in records) or "(no sources)"
    instructions = (SYSTEM + " Use only the sources below. If several revisions cover the asked "
                    "branch, follow the highest revision. After the answer, cite the source ID in "
                    "square brackets, for example: reservation [south-1]. If no source covers the "
                    "asked branch and topic, reply unknown.\n\nSources:\n" + sources)
    return [{"role":"system","content":instructions}, {"role":"user","content":question}]

def parse_answer(text, sources=()):
    lowered = text.casefold()
    labels = {label for key, label in (("walk-in","walk_in"), ("walk in","walk_in"),
                                       ("reservation","reservation")) if key in lowered}
    if "unknown" in lowered or len(labels) != 1:
        return verdict(None)
    cited = [source for source in dict.fromkeys(sources) if f"[{source}]" in lowered]
    return verdict(labels.pop(), cited[0] if len(cited) == 1 else None)

# Training prompts are worded differently from every holdout question.
OLD_TEMPLATES = ("Is a booking required for study rooms at {B}?",
                 "{B} branch: are study rooms walk-in or reservation?",
                 "What is the rule for using a study room at the {B} library?")

def old_snapshot_examples():
    return [(closed_book_messages(template.format(B=branch.capitalize())), label.replace("_", "-"))
            for branch, label in OLD.items() for template in OLD_TEMPLATES]

assert parse_answer("Reservation [east-2]", ["east-1","east-2"]) == verdict("reservation","east-2")
assert parse_answer("walk-in") == verdict("walk_in")
assert parse_answer("Unknown.") == verdict(None)
assert parse_answer("walk-in or reservation") == verdict(None)
assert len(old_snapshot_examples()) == 12
assert {answer for _, answer in old_snapshot_examples()} == {"walk-in","reservation"}
held_out_questions = {question for _, question, *_ in HOLDOUT}
assert not held_out_questions & {m[-1]["content"] for m, _ in old_snapshot_examples()}
print("OK")


In [ ]:
# Colab already has torch. Versions as of late 2026: dtype= needs transformers 4.56 or newer.
!pip install -q "transformers>=4.56" "peft>=0.17" "accelerate>=1.0"


In [ ]:
import random
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available(), "Switch to a GPU runtime (Runtime > Change runtime type > T4 GPU)"
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
torch.manual_seed(0)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# float32 avoids fp16 overflow during training; 0.5B parameters is about 2 GB, well within a T4.
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32).to("cuda")
model.eval()

@torch.no_grad()
def generate(messages, max_new_tokens=24):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                            pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def llm_closed_book(question, branch, topic):
    return parse_answer(generate(closed_book_messages(question)))

def llm_rag(question, branch, topic, k=3):
    records = policy_search(question, k)
    return parse_answer(generate(rag_messages(question, records)), [row["source"] for row in records])

print(generate(closed_book_messages(HOLDOUT[0][1])))


In [ ]:
# Evaluate the frozen model before any adapter is attached.
llm_report, llm_details = run_report(HOLDOUT, {"base": llm_closed_book, "rag_base": llm_rag})
for row in llm_details:
    print(row)


### Fine-tune with LoRA on the old snapshot

The adapter trains rank-8 updates on the attention projections while the 0.5B base weights stay
frozen. Only the answer tokens are scored: prompt positions get label `-100`, which the loss
ignores. The 12 training examples come from `OLD` only, so the adapter never sees the East and
West revisions or any holdout answer.


In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, task_type="CAUSAL_LM",
                         target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

def encode_example(messages, answer):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(answer + tokenizer.eos_token, add_special_tokens=False)["input_ids"]
    ids = torch.tensor([prompt_ids + answer_ids], device=model.device)
    labels = torch.tensor([[-100] * len(prompt_ids) + answer_ids], device=model.device)
    return ids, labels

EPOCHS = 10
examples = [encode_example(messages, answer) for messages, answer in old_snapshot_examples()]
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=3e-4)
model.train()
for epoch in range(EPOCHS):
    random.Random(epoch).shuffle(examples)
    total = 0.0
    for ids, labels in examples:
        loss = model(input_ids=ids, labels=labels).loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total += loss.item()
    print(f"epoch {epoch}: mean loss {total / len(examples):.4f}")
model.eval()

fit = [llm_closed_book(messages[-1]["content"], None, None)["label"] == answer.replace("-", "_")
       for messages, answer in old_snapshot_examples()]
print(f"Old-snapshot fit: {sum(fit)}/{len(fit)}. If this is low, raise EPOCHS and rerun from the load cell.")


In [ ]:
lora_report, lora_details = run_report(HOLDOUT, {"lora_llm": llm_closed_book})
llm_report.update(lora_report)
llm_details += lora_details

slices = ("all", "stable", "changed", "new_branch", "no_policy")
print("counts are [attempted, action, factual, citation]")
for title, table, systems in (("Offline toy", report, ("frozen","lora","rag")),
                              ("Qwen2.5-0.5B", llm_report, ("base","lora_llm","rag_base"))):
    print(title)
    for system in systems:
        print(f"  {system:9}", "  ".join(f"{s}={table.get((system, s))}" for s in slices))


## Submit your comparison

- Report each system's action, factual and citation counts with denominators, for both the
  offline toy and the GPU run, including the changed and missing-policy slices. Say which parts
  you ran and which you did not (MCP transport, GPU milestone).
- Explain why a LoRA adapter can fit the old snapshot yet return a stale answer after an update.
  Compare the toy adapter with `lora_llm`: where do they agree, and where does the real model differ?
- Inspect one stale or unsupported prediction from the GPU run. Cite the current policy chunk and
  trace the decision back to the old training snapshot, a retrieval miss, a stale revision or the
  answer step (`rag_messages` and the raw generation).
- Run one retrieval ablation and report its effect on both `rag` and `rag_base`: `k=1`, removing
  `STOPWORDS`, or withholding a public revision. Optionally fuse BM25 with a second ranking using
  reciprocal rank fusion. State what the experiment cannot tell you at this scale (7 holdout cases).
- Describe a release gate, an abstention policy and what you would record in a privacy-aware trace.
